Earlier in spring training, Cardinals top prospect JJ Weatherholt had no hits but had walked in 
half of his at bats. I was thinking about the extreme edges of player batting values and trying to understand 
how it would compare to more "normal" performances.

On one hand, a .000/.500/.000 slash line is an OPS of .500. That would normally be very bad. 
But on the other hand, getting on base half of the time would be an outstanding result that 
would lead to a lot of run scoring, even without any power.

There's not a simple way to calculate that with any accuracy, so I decided to build my own calculator, 
based on Fangraphs.com's formulas for wRC+.

## How wRC+ Is Calculated
wRC+ is a measure of hitter output that attempts to filter out all factors but what the batter himself did.
It's scaled with 100 as an average. So an average hitter has a 100 wRC+. A good hitter might have a 120 
wRC+, while an elite hitter could be quite a bit higher Aaron Judge led the majors with a 204 wRC+ in 2025. 
Ke'Bryan Hayes was last among quaifying hitters with a 65 wRC+.

At a high level, the stat is calculated by adding up the expected value of the outcome of each player at bat. 
(I say "expected" because the stat attempts to filter out the reliance on teammate's performance that you you have
with stats like RBIs.) Then that value is adjusted for park factors – how each home stadium affects batter 
outcomes. Finally, it's compared to the leage average hitter to see how much better or worse that batter's net outcome is.

## Inputs
The calculation start by adding up the batter's standard counting stats. You can fiddle with these numbers, and 
nothing else should need changing (with some caveats noted below).

In [129]:
YEAR = 2025
TEAM = 'Cardinals'
LEAGUE = 'NL'
AB = 388 # At Bats
BB = 43 # Walks
IBB = 1 # Intentional walks
HBP = 15 # Hit by pitch
SINGLES = 78 # Singles
DOUBLES = 13 # Doubles
TRIPLES = 0 # Triples
HOMERS = 19 # Home runs
SF = 4 # Sacrifice flies

## Data Setup
To make the calculations, I needed the weighted values for each plate appearance result.

First, I exported the data from Fangraphs's [Guts!](https://www.fangraphs.com/tools/guts?type=cn) page.

Secondly, I downloaded the 2025 Park Factors data. There isn't a single place to get every year's park
factors at a single shot. I didn't feel like scrolling back and getting every single year's factors, but I
think 2025 should be close enough to work as a rule of thumb for modern baseball.

Fangraphs expresses park factors as a perceentage, where 100 is average. But the calculations l
ater require it to be a decimal, so I divide it by 100 right here.

In [130]:
import pandas as pds

UBB = BB - IBB
PA = AB + BB - IBB + SF + HBP

league_weights = pds.read_csv('./data/fangraphs-guts-data-leaguewide-weights-by-year.csv')
year_weights = league_weights.loc[league_weights['Season'] == YEAR].reset_index(drop=True)

park_factors = pds.read_csv('./data/fangraphs-guts-data-park-factors-2025.csv')
team_park_factor = park_factors.loc[park_factors['Team'] == TEAM].reset_index(drop=True)
PF = team_park_factor['Basic (5yr)'] / 100

In [131]:
print(year_weights)
print("\nPARK FACTORS:")
print(team_park_factor)

   Season      wOBA  wOBAScale       wBB      wHBP       w1B      w2B  \
0    2025  0.313055    1.23167  0.691497  0.722289  0.882406  1.25191   

       w3B     wHR  runSB     runCS      R/PA      R/W      cFIP  
0  1.58446  2.0374    0.2 -0.409517  0.118158  9.77398  3.135136  

PARK FACTORS:
   Season       Team  Basic (5yr)        3yr        1yr          1B  \
0    2025  Cardinals    97.500134  99.491233  95.834076  101.070714   

          2B         3B         HR         SO         BB          GB  \
0  98.618948  88.620692  93.929237  97.267342  96.757519  100.878978   

           FB         LD        IFFB        FIP  
0  101.298451  99.400115  103.166687  97.936028  


# wOBA Calculation
[wOBA](https://library.fangraphs.com/offense/woba/) (weighted On Base Average) is a measure of batter output, with adjustments for how 
valuable each outcome is. Then it's adjusted to align with league-wide on-base percentage.
That makes it relatively easy to eyeball and decide if a number is good or not.

In [132]:
wOBA = (
    UBB * year_weights['wBB'] + 
    HBP * year_weights['wHBP'] + 
    SINGLES * year_weights['w1B'] +
    DOUBLES * year_weights['w2B'] +
    TRIPLES * year_weights['w3B'] +
    HOMERS * year_weights['wHR']
) / (PA)
print(f"wOBA: {wOBA}")

wOBA: 0    0.364566
dtype: float64


## wRAA Calculation
wRAA is the player's weighted runs above average. It starts from the wOBA calculation and converts
the decimal value to runs above the average hitter.

In [133]:
wRAA = ((wOBA - year_weights['wOBA']) / year_weights['wOBAScale']) * PA
print(f"wRAA: {wRAA}")

wRAA: 0    18.778254
dtype: float64


## wRC Calculation
wRC represents weighted runs created. This is different from wRAA, which compares itself
to the average major league player. wRC starts from a different 0-value: the [replacement player](https://library.fangraphs.com/misc/war/replacement-level/).
A replacement player is a theoretical borderline player who is readily available in any team's
minor league system who migh make the major leagues at times but won't provide much value over
any other minor leaguer that every other team has in their organization as well.

In [134]:
wRC = (
        ((wOBA - year_weights['wOBA']) / year_weights['wOBAScale']) +
        year_weights['R/PA']
) * PA
print(f"wRC: {wRC}")

wRC: 0    71.831196
dtype: float64


## Finally, wRC+
This is the gold standard: now that we know how many runs the batter created, we can
understand how much value they provide per at-bat and compare that to every other 
hitter in the league.

### The Caveats
#### AL/NL wRC/PA Excluding Pitchers
Fangraphs is annoyingly vague about one particular number: the [wRC+ formula
writeup](https://library.fangraphs.com/offense/wrc/) says that one of the 
necessary factors in the formula is "AL or NL wRC/PA excluding pitchers". But that number
is not included in the [Guts! page](https://www.fangraphs.com/tools/guts?type=cn)
where all of the other factors are. I calculated it manuallu for 2025 and 
hardcoded those numbers below. You would have to re-do this for every year that you
wanted to calculate, if you cared to work with other seasons than 2025.

#### The Actual wRC+ Formula
Up to this point, I've been entering Ivan Herrera's 2025 stats and comparing them to
what Fangraphs has to ensure my formulas are right. So far they have all matched exactly.
But the wRC+ formula as documented by Fangraphs does not work out. They claim the formula is:
```
wRC+ = (
    (
        (wRAA/PA + League R/PA) + (League R/PA – Park Factor* League R/PA)
    )/ (AL or NL wRC/PA excluding pitchers)
)*100`
```
Unless I'm doing something terribly wrong, that formula as written returns nonsense.
For example, Herrera's 2025 wRC+ comes out to -9387.503348.

I rewrote the formula to be conceptually correct, but now it doesn't match
with Fangraphs exactly. For example, Herrera's FG wRC+ is 137, while 
my calculation returns 138. But it's way closer than that junk above!

In [135]:
# Hardcoding my league calculations
AL_WRC_PA = 0.1178565558
NL_WRC_PA = 0.1184992854

# Choose the correct one per league
league_wrc_per_pa = NL_WRC_PA if LEAGUE == 'NL' else AL_WRC_PA

wRC_with_park_factor = wRC / PF
wRC_pf_per_pa = wRC_with_park_factor / PA
wRCPlus = wRC_pf_per_pa / league_wrc_per_pa * 100
print(f"wRC+: {wRCPlus}")

wRC+: 0    138.466851
dtype: float64


## Conclusion
Whew, that turned out to be a lot. Now that I have the calculator working, I'll 
address hypotheticals a future post.